# 03 – Diagnóstico × prognóstico (verificação do pipeline)

**Pergunta:** os mesmos dados, features e folds conseguem **reconhecer uma falha em curso** (diagnóstico)? 

Por que isso importa: na exploração, vários sinais mudaram de comportamento *durante* as falhas. Se o pipeline não reconhece nem a falha em curso, o problema está nas features, nos folds ou no código. Se reconhece bem, mas não antecipa (notebooks 02), o resultado é outro: o que muda aparece só quando a falha já começou, e isso é uma conclusão sobre os dados, não sobre o pipeline.

- Rótulo: `1` = minuto **dentro** de uma falha; `0` = normal. Sem horizonte.
- Mesmos folds por falha (forward-chaining, embargo de 24 h), mesma Silver de 1 min, mesmo baseline de acaso por rotação circular.
- Duas famílias de features: **pontuais** (valores do minuto) e **janelas** (1 h e 6 h, só passado, cobertura mínima).
- Métricas: lift (PR-AUC / prevalência), recall nos minutos de falha, **atraso** do primeiro alarme após o início da falha e falsos alarmes por dia.

In [1]:
import json
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import config
from evaluation import circular_shift_null, event_metrics, fixed_fpr_threshold
from splits import forward_chaining_folds

WINDOWS = [60, 360]       # minutos (1 h e 6 h)
MIN_COVERAGE = 0.8
FPR_TARGET = 0.01
EMBARGO = 24              # horas
N_TREES = 200
N_SHIFTS = 200
TRAIN_EVAL_ROWS = 100_000
SEED = config.SEED


## 1. Falhas, folds e Silver

In [2]:
rep = pd.read_csv(config.FAILURE_REPORT, parse_dates=["start_time", "end_time"]).sort_values("start_time")
failures = [
    {"id": int(r.failure_id), "start": r.start_time.to_pydatetime(), "end": r.end_time.to_pydatetime()}
    for r in rep.itertuples()
]
fail_start = {f["id"]: pd.Timestamp(f["start"]) for f in failures}
folds = forward_chaining_folds(failures, EMBARGO)

silver_dir = config.SILVER_DIR / "e1"
groups = json.loads((silver_dir / "_columns.json").read_text(encoding="utf-8"))
analog, digital = groups["analog"], groups["digital"]
sensors = analog + digital

sil = pd.read_parquet(silver_dir)
sil["ts"] = pd.to_datetime(sil["ts"]).dt.tz_localize(None)
sil = sil.sort_values("ts").reset_index(drop=True)

sil["label"] = 0
for f in failures:
    sil.loc[(sil["ts"] >= pd.Timestamp(f["start"])) & (sil["ts"] <= pd.Timestamp(f["end"])), "label"] = 1
print(f"{len(sil):,} minutos | em falha: {int(sil.label.sum()):,} ({sil.label.mean():.2%})")
print(sil.groupby(pd.cut(sil["ts"], [pd.Timestamp(0)] + [pd.Timestamp(f['end']) for f in failures] + [sil['ts'].max() + pd.Timedelta(1, 'D')])).label.sum())

252,382 minutos | em falha: 4,954 (1.96%)
ts
(1970-01-01 00:00:00, 2020-04-18 23:59:00]    1434
(2020-04-18 23:59:00, 2020-05-30 06:00:00]     391
(2020-05-30 06:00:00, 2020-06-07 14:30:00]    2861
(2020-06-07 14:30:00, 2020-07-15 19:00:00]     268
(2020-07-15 19:00:00, 2020-09-02 03:59:00]       0
Name: label, dtype: int64


## 2. Features (pontuais e em janelas por tempo)

In [3]:
s = sil.set_index("ts")
parts = [s[sensors]]
coverage_ok = pd.Series(True, index=s.index)
for w in WINDOWS:
    roll = s[sensors].rolling(f"{w}min")
    mean = roll.mean()
    mean.columns = [f"{c}_mean_{w}m" if c in analog else f"{c}_duty_{w}m" for c in sensors]
    std = s[analog].rolling(f"{w}min").std()
    std.columns = [f"{c}_std_{w}m" for c in analog]
    n = s["n_readings"].rolling(f"{w}min").count()
    coverage_ok &= (n / w) >= MIN_COVERAGE
    parts += [mean, std]

feat = pd.concat(parts, axis=1)
feat = feat.loc[:, ~feat.columns.duplicated()]
data = feat.join(s[["label"]]).reset_index()
data["coverage_ok"] = coverage_ok.to_numpy()

point_cols = sensors
window_cols = [c for c in feat.columns if c not in sensors]
FEATURE_SETS = {"pontuais": point_cols, "janelas": point_cols + window_cols}
print({k: len(v) for k, v in FEATURE_SETS.items()}, "| minutos com cobertura ok:", int(data.coverage_ok.sum()))

# População de avaliação comum: minutos com cobertura nas janelas
eval_mask = data["coverage_ok"]
for name, cols in FEATURE_SETS.items():
    sub = data[eval_mask] if name == "janelas" else data
    assert not sub[cols].isna().any().any(), f"{name}: nulos nas features"


{'pontuais': 15, 'janelas': 59} | minutos com cobertura ok: 216560


## 3. Modelos e avaliação por fold

In [4]:
def make_models():
    return {
        "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "RandomForest": RandomForestClassifier(
            n_estimators=N_TREES, max_depth=14, min_samples_leaf=20, n_jobs=-1, random_state=SEED),
        "LightGBM": LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=SEED, verbose=-1, n_jobs=-1),
    }

rows, preds = [], {}
for fname, cols in FEATURE_SETS.items():
    base = data[eval_mask] if fname == "janelas" else data
    for fold in folds:
        train = base[base["ts"] <= fold.train_end]
        test = data[eval_mask & (data["ts"] > fold.test_start) & (data["ts"] <= fold.test_end)]
        if train["label"].sum() == 0 or test["label"].sum() == 0:
            print(f"{fname} {fold.name}: sem positivos; pulando"); continue
        for mname, model in make_models().items():
            model.fit(train[cols], train["label"])
            t = test[["ts", "label"]].copy().reset_index(drop=True)
            t["score"] = model.predict_proba(test[cols])[:, 1]
            thr = fixed_fpr_threshold(t["score"], t["label"], FPR_TARGET)
            m = event_metrics(t, "score", thr, fail_start[fold.test_failure_id])
            pr = average_precision_score(t["label"], t["score"])
            prev = t["label"].mean()
            tr = train.sample(min(len(train), TRAIN_EVAL_ROWS), random_state=SEED)
            pr_train = average_precision_score(tr["label"], model.predict_proba(tr[cols])[:, 1])
            null = circular_shift_null(t, "score", FPR_TARGET, n_shifts=N_SHIFTS, seed=SEED)
            alarmed = t["score"] >= thr
            rows.append({
                "features": fname, "modelo": mname, "falha_testada": fold.test_failure_id,
                "minutos_teste": len(t), "minutos_em_falha": int(t["label"].sum()),
                "prevalencia": prev, "pr_auc": pr, "lift": pr / prev,
                "pr_auc_treino": pr_train, "p_pr_auc": (null["pr_auc"] >= pr).mean(),
                "recall_nos_minutos_de_falha": alarmed[t["label"] == 1].mean(),
                "detectada": m["detected"], "deteccao_por_acaso": null["detected"].mean(),
                "atraso_h": -m["lead_h"] if not np.isnan(m["lead_h"]) else np.nan,
                "false_alarms_per_day": m["false_alarms_per_day"], "limiar": thr,
            })
            preds[(fname, mname, fold.test_failure_id)] = (t, thr)
        print(fname, fold.name, "concluído")

res = pd.DataFrame(rows)
res.round(3)

pontuais fold_falha_2 concluído
pontuais fold_falha_3 concluído
pontuais fold_falha_4 concluído
janelas fold_falha_2 concluído
janelas fold_falha_3 concluído
janelas fold_falha_4 concluído


,features,modelo,falha_testada,minutos_teste,minutos_em_falha,prevalencia,pr_auc,lift,pr_auc_treino,p_pr_auc,recall_nos_minutos_de_falha,detectada,deteccao_por_acaso,atraso_h,false_alarms_per_day,limiar
0,pontuais,LogReg,2,36455,391,0.011,0.086,8.035,0.988,0.035,0.000,0.0,0.040,NaN,0.040,1.000
1,pontuais,RandomForest,2,36455,391,0.011,0.905,84.363,0.994,0.000,0.954,1.0,0.040,-0.000,0.119,0.669
2,pontuais,LightGBM,2,36455,391,0.011,0.141,13.152,0.996,0.005,0.077,1.0,0.045,0.967,0.040,1.000
3,pontuais,LogReg,3,8390,2527,0.301,0.939,3.117,0.805,0.000,0.458,1.0,0.985,0.083,0.515,0.772
4,pontuais,RandomForest,3,8390,2527,0.301,0.984,3.267,0.993,0.000,0.993,1.0,0.350,-0.000,0.687,0.025
5,pontuais,LightGBM,3,8390,2527,0.301,0.949,3.151,0.998,0.000,0.518,1.0,0.350,-0.000,1.716,0.000
6,pontuais,LogReg,4,38718,268,0.007,0.571,82.564,0.923,0.000,0.701,1.0,0.340,0.083,2.789,0.015
7,pontuais,RandomForest,4,38718,268,0.007,0.703,101.508,0.997,0.000,0.728,1.0,0.130,0.583,1.674,0.018
8,pontuais,LightGBM,4,38718,268,0.007,0.316,45.686,1.000,0.000,0.582,1.0,0.145,0.083,1.079,0.000
9,janelas,LogReg,2,36455,391,0.011,0.090,8.425,1.000,0.015,0.000,0.0,0.010,NaN,0.040,1.000


## 4. Resultados

Compare com o notebook 02 (prognóstico):
- Lift bem acima de 1 e `p_pr_auc` baixo aqui, mas lift ~1 no prognóstico: os sinais existem **durante** a falha e não antes.
- Lift ~1 aqui também: investigar features, folds ou código antes de concluir qualquer coisa.
- `atraso_h` é o tempo entre o início da falha e o primeiro alarme (quanto menor, melhor).

In [5]:
per_fold = res.pivot_table(index=["features", "modelo"], columns="falha_testada", values="lift").round(2)
per_fold.columns = [f"lift_falha_{c}" for c in per_fold.columns]
per_fold

lift_falha_2  lift_falha_3  lift_falha_4
features modelo                                                
janelas  LightGBM             11.91          3.31         48.86
         LogReg                8.43          3.26          1.57
         RandomForest          9.48          3.27         69.87
pontuais LightGBM             13.15          3.15         45.69
         LogReg                8.03          3.12         82.56
         RandomForest         84.36          3.27        101.51

In [6]:
summary = (
    res.groupby(["features", "modelo"])
    .agg(
        falhas_testadas=("falha_testada", "nunique"),
        falhas_detectadas=("detectada", "sum"),
        deteccoes_esperadas_por_acaso=("deteccao_por_acaso", "sum"),
        lift_medio=("lift", "mean"),
        p_pr_auc_max=("p_pr_auc", "max"),
        recall_minutos_falha=("recall_nos_minutos_de_falha", "mean"),
        atraso_h=("atraso_h", "mean"),
        pr_auc_treino=("pr_auc_treino", "mean"),
        falsos_alarmes_dia=("false_alarms_per_day", "mean"),
    )
    .round(3)
)
summary

falhas_testadas  falhas_detectadas  \
features modelo                                             
janelas  LightGBM                    3                3.0   
         LogReg                      3                2.0   
         RandomForest                3                2.0   
pontuais LightGBM                    3                3.0   
         LogReg                      3                2.0   
         RandomForest                3                3.0   

                       deteccoes_esperadas_por_acaso  lift_medio  \
features modelo                                                    
janelas  LightGBM                              0.445      21.362   
         LogReg                                0.550       4.421   
         RandomForest                          0.735      27.539   
pontuais LightGBM                              0.540      20.663   
         LogReg                                1.365      31.239   
         RandomForest                          0.520      63.046   

                       p_pr_auc_max  recall_minutos_falha  atraso_h  \
features modelo                                                       
janelas  LightGBM             0.010                 0.574     2.239   
         LogReg               0.105                 0.251     2.217   
         RandomForest         0.015                 0.454     1.175   
pontuais LightGBM             0.005                 0.392     0.350   
         LogReg               0.035                 0.386     0.083   
         RandomForest         0.000                 0.892     0.194   

                       pr_auc_treino  falsos_alarmes_dia  
features modelo                                           
janelas  LightGBM              1.000               0.485  
         LogReg                0.989               0.388  
         RandomForest          1.000               0.289  
pontuais LightGBM              0.998               0.945  
         LogReg                0.905               1.115  
         RandomForest          0.995               0.826

In [7]:
out = config.DOCS_DIR / "resultados_diagnostico.csv"
config.DOCS_DIR.mkdir(exist_ok=True)
res.to_csv(out, index=False)
print("Resultados por fold salvos em", out)

Resultados por fold salvos em C:\workspace\Projeto-Integrador-CienciaDados-UTFPR\docs\resultados_diagnostico.csv


## 5. Anotações para o artigo (preencher)

- O pipeline reconhece a falha em curso (lift alto, `p_pr_auc` baixo, recall alto)? Em todas as falhas ou só em algumas (a falha 4 não mostrou aumento de DV_pressure)?
- Qual o atraso médio do primeiro alarme após o início da falha?
- Contraste com o prognóstico (notebooks 02): o que muda só depois que a falha começa?
- Se o diagnóstico também falhar: investigar features, folds e código antes de qualquer conclusão.